In [ ]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox

# ---------------- DATABASE ----------------
conn = sqlite3.connect("sfoqs.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customerID INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT not null
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    orderID INTEGER PRIMARY KEY AUTOINCREMENT,
    customerID INTEGER,
    foodItem TEXT NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY(customerID) REFERENCES customers(customerID)
)
""")

conn.commit()

# ---------------- FUNCTIONS ----------------

def add_customer(name):
    cursor.execute("INSERT INTO customers (name) VALUES (?)", (name,))
    conn.commit()
    return cursor.lastrowid

def place_order():
    name = entry_name.get().strip()
    food = entry_food.get().strip()

    if not name or not food:
        messagebox.showerror("Error", "All fields are required")
        return

    try:
        customer_id = add_customer(name)

        cursor.execute("""
        INSERT INTO orders (customerID, foodItem, status)
        VALUES (?, ?, ?)
        """, (customer_id, food, "Pending"))

        conn.commit()

        messagebox.showinfo("Success", "Order placed successfully")

        entry_name.delete(0, tk.END)
        entry_food.delete(0, tk.END)

        load_orders()

    except Exception as e:
        messagebox.showerror("Database Error", str(e))

def load_orders():
    for row in tree.get_children():
        tree.delete(row)

    cursor.execute("""
    SELECT orders.orderID, customers.name, orders.foodItem, orders.status
    FROM orders
    JOIN customers ON orders.customerID = customers.customerID
    ORDER BY orders.orderID
    """)

    for row in cursor.fetchall():
        tree.insert("", tk.END, values=row)

def update_status(new_status):
    selected = tree.selection()

    if not selected:
        messagebox.showwarning("Warning", "Please select an order")
        return

    try:
        order_id = tree.item(selected[0])['values'][0]

        cursor.execute("UPDATE orders SET status=? WHERE orderID=?",
                       (new_status, order_id))
        conn.commit()

        load_orders()

    except Exception as e:
        messagebox.showerror("Error", str(e))

# ---------------- GUI ----------------

root = tk.Tk()
root.title("Smart Food Ordering & Queue System")
root.geometry("800x600")

title_label = tk.Label(root, text="Smart Food Ordering System",
                       font=("Arial", 16, "bold"))
title_label.pack(pady=10)

# Input Frame
frame = tk.Frame(root)
frame.pack(pady=10)

tk.Label(frame, text="Customer Name").grid(row=0, column=0, padx=5, pady=5)
entry_name = tk.Entry(frame)
entry_name.grid(row=0, column=1, padx=5)

tk.Label(frame, text="Food Item").grid(row=1, column=0, padx=5, pady=5)
entry_food = tk.Entry(frame)
entry_food.grid(row=1, column=1, padx=5)

tk.Button(frame, text="Place Order", command=place_order)\
    .grid(row=2, columnspan=2, pady=10)

# Table
columns = ("Order ID", "Customer", "Food", "Status")
tree = ttk.Treeview(root, columns=columns, show="headings")

for col in columns:
    tree.heading(col, text=col)
    tree.column(col, anchor="center")

tree.pack(fill=tk.BOTH, expand=True, pady=10)

# Buttons Frame
btn_frame = tk.Frame(root)
btn_frame.pack(pady=10)

tk.Button(btn_frame, text="Preparing",
          command=lambda: update_status("Preparing")).grid(row=0, column=0, padx=5)

tk.Button(btn_frame, text="Ready",
          command=lambda: update_status("Ready")).grid(row=0, column=1, padx=5)

tk.Button(btn_frame, text="Refresh",
          command=load_orders).grid(row=0, column=2, padx=5)

# Load initial data
load_orders()

# Run app
root.mainloop()

# Close DB when app closes
conn.close()